In [1]:
from surprise import Dataset
from surprise import Reader
from surprise import SVD

from surprise.model_selection import train_test_split
from surprise.accuracy import rmse


In [2]:
import pandas as pd
mv=pd.read_excel(r"C:\Users\RohanS\Downloads\movie.xlsx")
rt=pd.read_csv(r"C:\Users\RohanS\Downloads\ratings.csv")


In [3]:
ratings_sample = rt.sample(
    n=500000,
    random_state=42
)
ratings_sample.head()

,userId,movieId,rating,timestamp
17679788,122270,8360,3.5,1335056824
7106385,49018,32,2.0,1000194636
12970708,89527,109374,3.5,1420536400
15426752,106704,1060,3.0,948576477
6934678,47791,1732,2.0,1137685703


In [4]:
reader = Reader(rating_scale=(0.5,5))
data = Dataset.load_from_df(
    ratings_sample[['userId','movieId','rating']],
    reader
)

In [5]:
trainset, testset = train_test_split(
    data,
    test_size=0.2,
    random_state=42
)


In [6]:
model = SVD()
model.fit(trainset)
predictions = model.test(testset)
rmse(predictions)

RMSE: 0.9205


np.float64(0.9204932080983812)

In [7]:
model.predict(
    uid=1,
    iid=109487
)

Prediction(uid=1, iid=109487, r_ui=None, est=np.float64(4.030242919498779), details={'was_impossible': False})

In [9]:
movie_lookup = mv[
    ['movieId','title']
]

In [13]:
def recommend_svd(user_id, n=10):

    movie_ids = mv['movieId'].unique()

    predictions = []

    for movie_id in movie_ids:

        pred = model.predict(
            user_id,
            movie_id
        )

        predictions.append(
            (movie_id, pred.est)
        )

    predictions = sorted(
        predictions,
        key=lambda x:x[1],
        reverse=True
    )

    top_movies = predictions[:n]

    rec_df = pd.DataFrame(
        top_movies,
        columns=['movieId','pred_rating']
    )

    return rec_df.merge(
        movie_lookup,
        on='movieId'
    )
recommend_svd(23)

,movieId,pred_rating,title
0,668,5.000000,Song of the Little Road (Pather Panchali)
1,50,5.000000,"Usual Suspects, The"
2,318,4.885419,"Shawshank Redemption, The"
3,1248,4.860037,Touch of Evil
4,1284,4.853525,"Big Sleep, The"
5,2300,4.850638,"Producers, The"
6,1212,4.839811,"Third Man, The"
7,1283,4.839364,High Noon
8,1221,4.834113,"Godfather: Part II, The"
9,7502,4.804341,Band of Brothers
